In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

metadata_df

,sample_title,sample type,cell type,patient diagnosis
samples,,,,
GSM4290875,HD01,primary human sample,purified circulating tumor cells,healthy donor
GSM4290876,HD02,primary human sample,purified circulating tumor cells,healthy donor
GSM4290877,HD03,primary human sample,purified circulating tumor cells,healthy donor
GSM4290878,HD04,primary human sample,purified circulating tumor cells,healthy donor
GSM4290879,HD05,primary human sample,purified circulating tumor cells,healthy donor
...,...,...,...,...
GSM4290951,SU2C21,primary human sample,purified circulating tumor cells,localized PDAC
GSM4290952,SU2C22,primary human sample,purified circulating tumor cells,localized PDAC
GSM4290953,SU2C23,primary human sample,purified circulating tumor cells,localized PDAC


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()
# df_combined.to_csv(f'{GEO_ID}_raw_counts_NCBI.csv', index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_94483/4076848676.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,sample type,cell type,patient diagnosis,DDX11L1,WASH7P,MIR6859-1,MIR1302-2HG,MIR1302-2,FAM138A,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM4290875,HD01,primary human sample,purified circulating tumor cells,healthy donor,13,4,0,1,0,1,...,1705,20,15,18,1555,509,5,1511,18,25
GSM4290876,HD02,primary human sample,purified circulating tumor cells,healthy donor,17,5,0,2,1,2,...,2409,29,19,29,1816,648,16,2165,35,33
GSM4290877,HD03,primary human sample,purified circulating tumor cells,healthy donor,22,6,0,1,0,3,...,2938,3,3,3,1411,390,8,2512,15,12
GSM4290878,HD04,primary human sample,purified circulating tumor cells,healthy donor,76,20,0,7,1,7,...,8778,6,6,6,5334,1354,13,8620,42,71
GSM4290879,HD05,primary human sample,purified circulating tumor cells,healthy donor,56,17,0,1,2,1,...,5268,7,4,9,3215,750,9,4315,43,34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM4290951,SU2C21,primary human sample,purified circulating tumor cells,localized PDAC,84,5,0,0,1,2,...,2669,24,22,15,1138,499,19,1764,13,20
GSM4290952,SU2C22,primary human sample,purified circulating tumor cells,localized PDAC,18,5,1,1,0,1,...,1637,15,12,12,1176,239,8,1431,8,8
GSM4290953,SU2C23,primary human sample,purified circulating tumor cells,localized PDAC,13,1,0,2,0,3,...,900,8,7,9,512,172,6,752,12,16


In [3]:
import pandas as pd

# Download supplementary data from: https://pmc.ncbi.nlm.nih.gov/articles/instance/7335061/bin/41467_2020_17150_MOESM4_ESM.xlsx

# Read specific sheet (sheet number 6, which is index 5)
df_clinical_data = pd.read_excel('41467_2020_17150_MOESM4_ESM.xlsx', sheet_name=5, skiprows=1)
df_clinical_data = df_clinical_data.iloc[:-2,:-5]
df_clinical_data.to_csv('Table_S4_clinical_data.csv', index=False)


df_combined = df_clinical_data.merge(
    df_combined,
    left_on='ID',
    right_on='sample_title',
    how='inner'
)


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.to_csv(data_file_name, index=False)

# Generate description


In [4]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, 'Table_S4_clinical_data.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': df_clinical_data.shape[0],
    'clinical_features': df_combined.shape[1] - df_raw_counts.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
